In [1]:
from pathlib import Path
import os
import shutil
import sys
import zipfile

SOURCE_ROOT = Path("/kaggle/input/datasets/huynhnhuthuyk18hcm/minimed-prime-source")
WORK_ROOT = Path("/kaggle/working/MiniMed_Prime")

def resolve_source_dir(root: Path, name: str):
    direct = root / name
    if not direct.is_dir():
        return None

    nested = direct / name
    signatures = {
        "src": "orchestrator.py",
        "scripts": "setup_environment.py",
    }
    sig = signatures.get(name)

    if sig:
        if (direct / sig).exists():
            return direct
        if (nested / sig).exists():
            return nested

    direct_entries = sorted(p.name for p in direct.iterdir())
    if nested.is_dir() and direct_entries == [name]:
        return nested

    return direct

shutil.rmtree(WORK_ROOT, ignore_errors=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

resolved = {}

for name in ["src", "scripts", "notebooks", "tests"]:
    zip_path = SOURCE_ROOT / f"{name}.zip"
    if zip_path.exists():
        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(WORK_ROOT)
        resolved[name] = f"{zip_path.name} -> extracted"
        continue

    source_dir = resolve_source_dir(SOURCE_ROOT, name)
    if source_dir is not None:
        shutil.copytree(source_dir, WORK_ROOT / name, dirs_exist_ok=True)
        try:
            resolved[name] = str(source_dir.relative_to(SOURCE_ROOT))
        except ValueError:
            resolved[name] = str(source_dir)

for file_name in [
    "requirements-integration.txt",
    "README_KAGGLE_VI.md",
    "TRAINING_KAGGLE_LOCAL_VI.md",
    "IMPLEMENTATION_AUDIT.md",
    "KAGGLE_SOURCE_VERSION.txt",
]:
    source_file = SOURCE_ROOT / file_name
    if source_file.exists():
        shutil.copy2(source_file, WORK_ROOT / file_name)

os.chdir(WORK_ROOT)
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

marker = WORK_ROOT / "KAGGLE_SOURCE_VERSION.txt"
print("source root:", SOURCE_ROOT)
print("resolved dirs:", resolved)
print("version marker exists:", marker.exists())
if marker.exists():
    print(marker.read_text(encoding="utf-8"))
print("setup_environment exists:", (WORK_ROOT / "scripts" / "setup_environment.py").exists())


source root: /kaggle/input/datasets/huynhnhuthuyk18hcm/minimed-prime-source
resolved dirs: {'src': 'src', 'scripts': 'scripts', 'notebooks': 'notebooks', 'tests': 'tests'}
version marker exists: True
MiniMed Prime Kaggle source package
version: 2026-04-20-ticket-017-trm-validation-and-nonstrict-mode
notes:
- Added detailed checkpoint inspection and state-dict compatibility diagnostics for Samsung TRM loading
- Added robust dummy forward-pass validation with input/output shape/type logging
- Added strict/non-strict TRM validation mode for emergency loading paths
- Wired trm_strict_validate through orchestrator and CLI scripts (smoke_test, verify_components, check_trm_ready)

setup_environment exists: True


In [2]:
!pip install rank_bm25

In [3]:
!nvidia-smi
!python scripts/setup_environment.py --kaggle --skip-downloads

Mon Apr 20 02:12:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from pathlib import Path
from src.utils.kaggle_env import KaggleEnv

print("has DATASET_SLUG_HINTS:", hasattr(KaggleEnv, "DATASET_SLUG_HINTS"))
print("DATASET_SLUG_HINTS:", getattr(KaggleEnv, "DATASET_SLUG_HINTS", None))
print("input dirs:", sorted(p.name for p in Path("/kaggle/input").iterdir()))

has DATASET_SLUG_HINTS: True
DATASET_SLUG_HINTS: {'primekg': ('primekg-medical-knowledge-graph',), 'medreason': ('medreason-medical-reasoning-dataset',), 'sapbert': ('sapbert-from-pubmedbert-fulltext',), 'medcpt-query': ('medcpt-query-encoder',), 'medcpt-cross': ('medcpt-cross-encoder',), 'medcpt-article': ('medcpt-article-encoder',), 'trm-real': ('tinyrecursivemodels-real-repo-and-arc', 'tinyrecursivemodels')}
input dirs: ['datasets']


In [5]:
import os
from pprint import pprint

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")
    print("Loaded OPENAI_API_KEY from Kaggle Secrets.")
except Exception as exc:
    print(f"OPENAI_API_KEY not loaded from Kaggle Secrets: {exc}")

os.environ["MINIMED_JUDGE_MODEL"] = "gpt-4o-mini"
os.environ["MEDREASON_EDGE_LLM"] = "gpt-4o-mini"
os.environ["MINIMED_SYNTHESIS_MODEL"] = "gpt-4o-mini"

pprint({
    "OPENAI_API_KEY": "set" if os.environ.get("OPENAI_API_KEY") else "missing",
    "MINIMED_JUDGE_MODEL": os.environ.get("MINIMED_JUDGE_MODEL"),
    "MEDREASON_EDGE_LLM": os.environ.get("MEDREASON_EDGE_LLM"),
    "MINIMED_SYNTHESIS_MODEL": os.environ.get("MINIMED_SYNTHESIS_MODEL"),
})


Loaded OPENAI_API_KEY from Kaggle Secrets.
{'MEDREASON_EDGE_LLM': 'gpt-4o-mini',
 'MINIMED_JUDGE_MODEL': 'gpt-4o-mini',
 'MINIMED_SYNTHESIS_MODEL': 'gpt-4o-mini',
 'OPENAI_API_KEY': 'set'}


In [6]:
from src.utils.kaggle_env import KaggleEnv

for p in [
    "data/kg/primekg",
    "data/checkpoints/sapbert",
    "data/checkpoints/medcpt-query",
    "data/checkpoints/medcpt-article",
    "data/checkpoints/medreason-8b",
    "external/TinyRecursiveModels",
]:
    print(p, "->", KaggleEnv.path(p))


data/kg/primekg -> /kaggle/input/datasets/huynhnhuthuyk18hcm/primekg
data/checkpoints/sapbert -> /kaggle/input/datasets/huynhnhuthuyk18hcm/sapbert
data/checkpoints/medcpt-query -> /kaggle/input/datasets/huynhnhuthuyk18hcm/medcpt-query
data/checkpoints/medcpt-article -> /kaggle/input/datasets/huynhnhuthuyk18hcm/medcpt-article
data/checkpoints/medreason-8b -> /kaggle/input/datasets/huynhnhuthuyk18hcm/medreason-8b
external/TinyRecursiveModels -> /kaggle/working/external/TinyRecursiveModels


In [7]:
!pip install scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_lg-0.5.4.tar.gz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 71.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 69.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 wh

In [8]:
!python scripts/verify_components.py --kaggle

[2026-04-20 02:15:30,575] minimed.layer3_trm INFO trm_init {"backend_used": "samsung_trm", "checkpoint_loaded": true, "checkpoint_path": "/kaggle/working/external/TinyRecursiveModels/checkpoints/trm_arc_v1_public_step_518071.pt", "input_remap_active": true, "is_real_checkpoint": true, "latency_ms": 0.0, "load_diagnostics": ["TRM path source selected: path_hint", "TRM repo: SAMSUNG structure (models/recursive_reasoning/trm.py)", "TRM check: repo_root_exists -> EXISTS", "TRM check: models/recursive_reasoning/trm.py -> EXISTS", "TRM check: config/arch/trm.yaml -> EXISTS", "TRM check: /kaggle/working/external/TinyRecursiveModels/checkpoints/trm_arc_v1_public_step_518071.pt -> EXISTS", "=== TRM Load Attempt ===", "Resolved checkpoint path: /kaggle/working/external/TinyRecursiveModels/checkpoints/trm_arc_v1_public_step_518071.pt", "Resolved repo root: /kaggle/working/external/TinyRecursiveModels", "Checkpoint size bytes: 1822205258", "TRM repo sample python files: ['/kaggle/working/external/

In [9]:
!python scripts/smoke_test.py --kaggle

[2026-04-20 02:15:59,760] minimed.layer3_trm INFO trm_init {"backend_used": "fallback_trm", "checkpoint_loaded": false, "checkpoint_path": "/kaggle/working/external/TinyRecursiveModels/checkpoints/trm_arc_v1_public_step_518071.pt", "input_remap_active": false, "is_real_checkpoint": false, "latency_ms": 0.0, "load_diagnostics": ["TRM path source selected: path_hint", "TRM repo: SAMSUNG structure (models/recursive_reasoning/trm.py)", "TRM check: repo_root_exists -> EXISTS", "TRM check: models/recursive_reasoning/trm.py -> EXISTS", "TRM check: config/arch/trm.yaml -> EXISTS", "TRM check: /kaggle/working/external/TinyRecursiveModels/checkpoints/trm_arc_v1_public_step_518071.pt -> EXISTS", "Unable to load official Samsung TRM. Falling back to internal wrapper: Loaded TRM checkpoint failed dummy forward-pass validation.", "Traceback (most recent call last):\n  File \"/kaggle/working/MiniMed_Prime/src/models/trm_wrapper.py\", line 605, in load_trm_model_bundle\n    bundle = load_real_trm(\n  

In [10]:
import json

# Open the file in read mode
with open("/kaggle/working/smoke_test_output.json", "r") as file:
    data = json.load(file)

# Pretty-print the JSON data
print(json.dumps(data, indent=4))

{
    "answer_text": "ABSTENTION: I cannot provide a confident medical answer because trm confidence too low [edge:4398|disease_phenotype_positive|12552_8234_8082_7540_19003_17169|5854220] [edge:4398|disease_phenotype_positive|8678|5911740].\nReasoning: The verified evidence bundle contains 200 KG edges and 0 PubMed passages, but the reasoning signal remained insufficient or contradictory [edge:4398|disease_phenotype_positive|12552_8234_8082_7540_19003_17169|5854220] [edge:4398|disease_phenotype_positive|8678|5911740].\nConfidence level: low (0.00) [edge:4398|disease_phenotype_positive|12552_8234_8082_7540_19003_17169|5854220] [edge:4398|disease_phenotype_positive|8678|5911740].",
    "question_id": "3fa5cfa8-a64b-470f-9676-8fe12427fa32",
    "provenance": [
        "edge:4398|disease_phenotype_positive|12552_8234_8082_7540_19003_17169|5854220",
        "edge:4398|disease_phenotype_positive|8678|5911740"
    ],
    "confidence": 0.0015359107637777925,
    "abstention": true,
    "abste